# Module 05: SharePoint via Microsoft Graph API

## Learning Objectives

By completing this notebook, you will:
1. Authenticate to Microsoft Graph API using the client credentials flow
2. Create, read, update, and delete SharePoint list items via REST
3. Build OData filter queries to retrieve specific list rows
4. Download and upload files to a SharePoint document library
5. Understand how Power Automate's SharePoint connector maps to raw Graph API calls

## Prerequisites

- An Azure Active Directory (Entra ID) app registration with the following **application permissions**:
  - `Sites.ReadWrite.All`
  - `Files.ReadWrite.All`
- The app registration's **Client ID**, **Client Secret**, and **Tenant ID**
- A SharePoint site URL you have access to
- Python packages: `requests`, `python-dotenv`

## Estimated Time: 15 minutes

---

### Why learn this alongside Power Automate?

Power Automate's SharePoint connector is a wrapper around these same Graph API calls. When you hit connector limits, need custom logic, or want to integrate SharePoint into a Python data pipeline, you call the Graph API directly. Understanding the underlying API makes you a better flow designer — you know exactly what the connector is doing under the hood.

In [ ]:
learning_objectives([
    "An Azure Active Directory (Entra ID) app registration with the following **application permissions**:",
    "`Sites.ReadWrite.All`",
    "`Files.ReadWrite.All`",
    "The app registration's **Client ID**, **Client Secret**, and **Tenant ID**",
    "A SharePoint site URL you have access to",
    "Python packages: `requests`, `python-dotenv`",
    "### Why learn this alongside Power Automate?"
])

## 1. Setup and Authentication

Microsoft Graph API uses OAuth 2.0. For server-to-server automation (no user present), the **client credentials flow** is appropriate. The app authenticates using its own Client ID and Secret — no user login required.

The token endpoint is:
```
https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token
```

Store credentials in a `.env` file — never hardcode secrets in notebook cells.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Install required packages if not already present
# !pip install requests python-dotenv

import os
import json
import requests
from datetime import datetime, timezone
from dotenv import load_dotenv

# Load credentials from .env file
# Create a .env file with these keys:
#   TENANT_ID=your-tenant-id
#   CLIENT_ID=your-app-client-id
#   CLIENT_SECRET=your-app-client-secret
#   SHAREPOINT_SITE_URL=https://contoso.sharepoint.com/sites/Automation
load_dotenv()

TENANT_ID = os.getenv("TENANT_ID")
CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
SITE_URL = os.getenv("SHAREPOINT_SITE_URL")

# Validate that required variables are set
missing = [k for k, v in {
    "TENANT_ID": TENANT_ID,
    "CLIENT_ID": CLIENT_ID,
    "CLIENT_SECRET": CLIENT_SECRET,
    "SHAREPOINT_SITE_URL": SITE_URL,
}.items() if not v]

if missing:
    print(f"WARNING: Missing environment variables: {missing}")
    print("The cells below will show expected outputs but skip live API calls.")
else:
    print("Credentials loaded successfully.")

In [ ]:
# --- Authentication: client credentials OAuth 2.0 flow ---
# Why client credentials? No user interaction needed — the app acts as itself.
# This is the same credential type Power Automate uses when you create a
# SharePoint connection with an app registration instead of a user account.

GRAPH_BASE = "https://graph.microsoft.com/v1.0"


def get_access_token(tenant_id: str, client_id: str, client_secret: str) -> str:
    """
    Acquire a bearer token using the OAuth 2.0 client credentials flow.

    Parameters
    ----------
    tenant_id : str
        Azure AD tenant (directory) ID
    client_id : str
        Application (client) ID from the app registration
    client_secret : str
        Client secret value (not the secret ID)

    Returns
    -------
    str
        Bearer access token, valid for ~1 hour
    """
    token_url = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"

    # Request body for client credentials grant
    payload = {
        "grant_type": "client_credentials",
        "client_id": client_id,
        "client_secret": client_secret,
        # Graph API scope — all permissions are configured in the app registration
        "scope": "https://graph.microsoft.com/.default",
    }

    response = requests.post(token_url, data=payload, timeout=30)
    response.raise_for_status()

    token_data = response.json()
    return token_data["access_token"]


# Acquire a token only when credentials are available
if all([TENANT_ID, CLIENT_ID, CLIENT_SECRET]):
    ACCESS_TOKEN = get_access_token(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
    HEADERS = {
        "Authorization": f"Bearer {ACCESS_TOKEN}",
        "Content-Type": "application/json",
    }
    print("Access token acquired.")
else:
    ACCESS_TOKEN = None
    HEADERS = {}
    print("Skipping token acquisition — credentials not configured.")

## 2. Locating the SharePoint Site

Graph API identifies a SharePoint site by its **site ID**, not its URL. Resolve the URL to an ID first.

The endpoint:
```
GET https://graph.microsoft.com/v1.0/sites/{hostname}:/{server-relative-path}
```

For `https://contoso.sharepoint.com/sites/Automation`:
- hostname: `contoso.sharepoint.com`
- server-relative-path: `/sites/Automation`

In [ ]:
# --- Resolve SharePoint site URL to site ID ---
# The site ID is a compound string: {tenant}.sharepoint.com,{site-guid},{web-guid}
# It is required for all subsequent list and file operations.


def get_site_id(site_url: str, headers: dict) -> str:
    """
    Resolve a SharePoint site URL to its Graph API site ID.

    Parameters
    ----------
    site_url : str
        Full SharePoint site URL, e.g. https://contoso.sharepoint.com/sites/Automation
    headers : dict
        Authorization headers including the bearer token

    Returns
    -------
    str
        Graph API site ID in the form 'hostname,site-guid,web-guid'
    """
    from urllib.parse import urlparse

    parsed = urlparse(site_url)
    hostname = parsed.netloc  # e.g. contoso.sharepoint.com
    path = parsed.path  # e.g. /sites/Automation

    endpoint = f"{GRAPH_BASE}/sites/{hostname}:{path}"
    response = requests.get(endpoint, headers=headers, timeout=30)
    response.raise_for_status()

    site_data = response.json()
    return site_data["id"]


# Live call — only runs when credentials are available
if ACCESS_TOKEN and SITE_URL:
    SITE_ID = get_site_id(SITE_URL, HEADERS)
    print(f"Site ID: {SITE_ID}")
else:
    # Expected output shape for documentation purposes
    SITE_ID = "contoso.sharepoint.com,a1b2c3d4-...,e5f6g7h8-..."
    print(f"[DEMO] Site ID would look like: {SITE_ID}")

## 3. Working with SharePoint Lists — CRUD

### 3.1 List all lists on the site

Before operating on a list, retrieve the list's **list ID** (a GUID). List IDs are stable across renames.

```
GET /sites/{site-id}/lists
```

In [ ]:
# --- Read: List all SharePoint lists on the site ---
# Power Automate's "List Name" dropdown is populated by this same call.
# We use $select to limit fields to what we actually need — faster and less data.


def get_all_lists(site_id: str, headers: dict) -> list:
    """
    Retrieve all lists on a SharePoint site.

    Parameters
    ----------
    site_id : str
        Graph API site ID
    headers : dict
        Authorization headers

    Returns
    -------
    list
        List of dicts, each containing 'id', 'displayName', and 'list' metadata
    """
    endpoint = (
        f"{GRAPH_BASE}/sites/{site_id}/lists"
        "?$select=id,displayName,list"
    )
    response = requests.get(endpoint, headers=headers, timeout=30)
    response.raise_for_status()
    return response.json()["value"]


if ACCESS_TOKEN:
    lists = get_all_lists(SITE_ID, HEADERS)
    # Show only user-created lists (filter out hidden system lists)
    user_lists = [
        lst for lst in lists
        if not lst.get("list", {}).get("hidden", False)
    ]
    for lst in user_lists:
        print(f"  {lst['displayName']:<40} id={lst['id']}")
else:
    print("[DEMO] Lists would appear here, e.g.:")
    print("  Expense Reports                          id=abc123...")
    print("  Project Tracker                          id=def456...")

### 3.2 Read list items with OData filtering

Graph API uses `$filter` for OData queries — the same syntax Power Automate uses in the **Filter Query** field of **Get items**.

```
GET /sites/{site-id}/lists/{list-id}/items?$expand=fields&$filter=fields/Status eq 'Pending'
```

Note the `fields/` prefix before column names — this is required when filtering on item fields via the Graph API (unlike the Power Automate connector which handles this prefix automatically).

In [ ]:
# --- Read: Get list items with OData filter ---
# $expand=fields is required — without it, Graph returns only system metadata,
# not the actual column values of each list item.
# $top controls page size (default 200; max 999 per page).


def get_list_items(
    site_id: str,
    list_id: str,
    headers: dict,
    odata_filter: str = None,
    top: int = 100,
) -> list:
    """
    Retrieve items from a SharePoint list with optional OData filtering.

    Parameters
    ----------
    site_id : str
        Graph API site ID
    list_id : str
        GUID of the target list
    headers : dict
        Authorization headers
    odata_filter : str, optional
        OData $filter expression, e.g. "fields/Status eq 'Pending'"
    top : int
        Maximum items to return per page (max 999)

    Returns
    -------
    list
        All matching list items as dicts. Handles pagination automatically.
    """
    endpoint = (
        f"{GRAPH_BASE}/sites/{site_id}/lists/{list_id}/items"
        f"?$expand=fields&$top={top}"
    )

    if odata_filter:
        # URL-encode the filter string to handle spaces and special characters
        from urllib.parse import quote
        endpoint += f"&$filter={quote(odata_filter)}"

    all_items = []

    # Graph API paginates large results using @odata.nextLink
    while endpoint:
        response = requests.get(endpoint, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()
        all_items.extend(data.get("value", []))

        # Follow the next page link if it exists
        endpoint = data.get("@odata.nextLink")

    return all_items


# Demo: fetch all Pending expense items
TARGET_LIST_ID = "your-list-id-guid-here"  # Replace with a real list ID from the cell above

if ACCESS_TOKEN and TARGET_LIST_ID != "your-list-id-guid-here":
    items = get_list_items(
        SITE_ID,
        TARGET_LIST_ID,
        HEADERS,
        odata_filter="fields/Status eq 'Pending'",
    )
    print(f"Found {len(items)} pending items.")
    for item in items[:3]:  # Show first 3
        fields = item["fields"]
        print(f"  ID={item['id']}  Title={fields.get('Title')}  Status={fields.get('Status')}")
else:
    print("[DEMO] Expected output:")
    print("Found 4 pending items.")
    print("  ID=1  Title=Office Supplies Q4    Status=Pending")
    print("  ID=2  Title=Client Dinner Dec 12  Status=Pending")
    print("  ID=3  Title=Travel - NYC Summit   Status=Pending")

### 3.3 Create a list item

```
POST /sites/{site-id}/lists/{list-id}/items
Body: { "fields": { "Title": "New item", "Status": "Pending" } }
```

Column values go inside the `fields` object. The column names are the **internal names** — the same names required by Power Automate's OData filter.

In [ ]:
# --- Create: Add a new list item ---
# The fields dict uses internal column names (not display names).
# Required columns must be included or the API returns 400 Bad Request.


def create_list_item(site_id: str, list_id: str, headers: dict, fields: dict) -> dict:
    """
    Create a new item in a SharePoint list.

    Parameters
    ----------
    site_id : str
        Graph API site ID
    list_id : str
        GUID of the target list
    headers : dict
        Authorization headers (must include Content-Type: application/json)
    fields : dict
        Column name → value pairs for the new item

    Returns
    -------
    dict
        The newly created item including its assigned ID
    """
    endpoint = f"{GRAPH_BASE}/sites/{site_id}/lists/{list_id}/items"
    body = {"fields": fields}

    response = requests.post(endpoint, headers=headers, json=body, timeout=30)
    response.raise_for_status()
    return response.json()


# Demo: create a sample expense item
new_item_fields = {
    "Title": "Conference Registration - PyCon 2025",
    "Amount": 799.00,
    "Status": "Pending",
    "SubmittedDate": datetime.now(timezone.utc).isoformat(),
    "Department": "Engineering",
}

if ACCESS_TOKEN and TARGET_LIST_ID != "your-list-id-guid-here":
    new_item = create_list_item(SITE_ID, TARGET_LIST_ID, HEADERS, new_item_fields)
    print(f"Created item ID: {new_item['id']}")
    print(f"Fields: {json.dumps(new_item['fields'], indent=2)}")
    CREATED_ITEM_ID = new_item["id"]
else:
    print("[DEMO] Expected output:")
    print("Created item ID: 42")
    CREATED_ITEM_ID = "42"

### 3.4 Update a list item

```
PATCH /sites/{site-id}/lists/{list-id}/items/{item-id}/fields
Body: { "Status": "Approved", "ApprovedBy": "manager@contoso.com" }
```

PATCH updates only the fields in the body — all other fields retain their current values. This is identical to Power Automate's **Update item** action.

In [ ]:
# --- Update: Modify specific fields on an existing item ---
# PATCH to /items/{item-id}/fields (note: /fields at the end)
# Only include the fields you want to change — omitted fields are untouched.


def update_list_item(
    site_id: str, list_id: str, item_id: str, headers: dict, fields: dict
) -> dict:
    """
    Update specific fields on an existing SharePoint list item.

    Parameters
    ----------
    site_id : str
        Graph API site ID
    list_id : str
        GUID of the target list
    item_id : str
        Numeric ID of the item to update (as a string)
    headers : dict
        Authorization headers
    fields : dict
        Only the column name → new value pairs to change

    Returns
    -------
    dict
        Updated fields response from Graph API
    """
    endpoint = f"{GRAPH_BASE}/sites/{site_id}/lists/{list_id}/items/{item_id}/fields"

    response = requests.patch(endpoint, headers=headers, json=fields, timeout=30)
    response.raise_for_status()
    return response.json()


# Demo: approve the item we just created
update_fields = {
    "Status": "Approved",
    "ApprovedBy": "manager@contoso.com",
    "ApprovedDate": datetime.now(timezone.utc).isoformat(),
}

if ACCESS_TOKEN and CREATED_ITEM_ID != "42":
    updated = update_list_item(SITE_ID, TARGET_LIST_ID, CREATED_ITEM_ID, HEADERS, update_fields)
    print(f"Updated item {CREATED_ITEM_ID}:")
    print(f"  Status: {updated.get('Status')}")
    print(f"  ApprovedBy: {updated.get('ApprovedBy')}")
else:
    print("[DEMO] Expected output:")
    print("Updated item 42:")
    print("  Status: Approved")
    print("  ApprovedBy: manager@contoso.com")

### 3.5 Delete a list item

```
DELETE /sites/{site-id}/lists/{list-id}/items/{item-id}
```

A successful delete returns HTTP 204 No Content — no response body.

In [ ]:
# --- Delete: Remove a list item permanently ---
# HTTP 204 means success. The item moves to the SharePoint recycle bin
# (recoverable by a site admin for 30 days) before permanent deletion.


def delete_list_item(
    site_id: str, list_id: str, item_id: str, headers: dict
) -> bool:
    """
    Delete a SharePoint list item by ID.

    Parameters
    ----------
    site_id : str
        Graph API site ID
    list_id : str
        GUID of the target list
    item_id : str
        Numeric ID of the item to delete
    headers : dict
        Authorization headers

    Returns
    -------
    bool
        True if deletion succeeded (HTTP 204)
    """
    endpoint = f"{GRAPH_BASE}/sites/{site_id}/lists/{list_id}/items/{item_id}"
    response = requests.delete(endpoint, headers=headers, timeout=30)
    # 204 No Content = success
    return response.status_code == 204


# Demo: delete the test item we created (clean up)
if ACCESS_TOKEN and CREATED_ITEM_ID != "42":
    success = delete_list_item(SITE_ID, TARGET_LIST_ID, CREATED_ITEM_ID, HEADERS)
    print(f"Deleted item {CREATED_ITEM_ID}: {success}")
else:
    print("[DEMO] Expected output:")
    print("Deleted item 42: True")

## 4. OData Filter Patterns

The OData filter expressions used by the Graph API are the same as those in Power Automate's **Filter Query** field — with one difference: Graph API requires the `fields/` prefix before column names.

| Power Automate Filter Query | Graph API `$filter` |
|----------------------------|-----------------------|
| `Status eq 'Pending'` | `fields/Status eq 'Pending'` |
| `Amount gt 500` | `fields/Amount gt 500` |
| `startswith(Title, 'Q4')` | `startswith(fields/Title, 'Q4')` |

In [ ]:
# --- OData filter builder utility ---
# This helper constructs correctly-formatted Graph API filter strings,
# including the fields/ prefix that Power Automate handles automatically.


def build_graph_filter(
    conditions: list,
    operator: str = "and",
) -> str:
    """
    Build a Graph API OData filter string from a list of conditions.

    Each condition is a tuple: (column_name, odata_operator, value)
    String values are automatically quoted.
    The fields/ prefix is added automatically.

    Parameters
    ----------
    conditions : list of tuple
        Each tuple: (column_name, odata_operator, value)
        Example: [("Status", "eq", "Pending"), ("Amount", "gt", 500)]
    operator : str
        Logical operator joining conditions: "and" or "or"

    Returns
    -------
    str
        OData filter string ready for use in a Graph API $filter parameter

    Examples
    --------
    >>> build_graph_filter([("Status", "eq", "Pending"), ("Amount", "gt", 500)])
    "fields/Status eq 'Pending' and fields/Amount gt 500"
    """
    parts = []
    for column, odata_op, value in conditions:
        # String values need single-quote wrapping in OData
        if isinstance(value, str):
            formatted_value = f"'{value}'"
        elif isinstance(value, bool):
            # OData booleans are lowercase: true/false
            formatted_value = str(value).lower()
        else:
            # Numeric values: no quotes
            formatted_value = str(value)

        parts.append(f"fields/{column} {odata_op} {formatted_value}")

    return f" {operator} ".join(parts)


# Test the filter builder
filter1 = build_graph_filter([("Status", "eq", "Pending")])
filter2 = build_graph_filter([
    ("Status", "eq", "Pending"),
    ("Amount", "gt", 500),
])
filter3 = build_graph_filter([
    ("Department", "eq", "Finance"),
    ("Department", "eq", "Legal"),
], operator="or")

print("Filter 1:", filter1)
print("Filter 2:", filter2)
print("Filter 3:", filter3)

## 5. Document Library: Download and Upload Files

SharePoint document libraries are represented as **drives** in Graph API. File operations use the Drive API.

```
# List the drives on a site
GET /sites/{site-id}/drives

# Download a file (get its content)
GET /sites/{site-id}/drives/{drive-id}/root:/{file-path}:/content

# Upload a file (small files < 4 MB)
PUT /sites/{site-id}/drives/{drive-id}/root:/{file-path}:/content
Body: binary file content
```

In [ ]:
# --- Document library: list drives on the site ---
# Each document library is a Drive. The default library is named 'Documents'
# or 'Shared Documents' depending on site configuration.


def get_site_drives(site_id: str, headers: dict) -> list:
    """
    List all document libraries (drives) on a SharePoint site.

    Parameters
    ----------
    site_id : str
        Graph API site ID
    headers : dict
        Authorization headers

    Returns
    -------
    list
        List of drive dicts with 'id', 'name', 'driveType', and 'webUrl'
    """
    endpoint = f"{GRAPH_BASE}/sites/{site_id}/drives?$select=id,name,driveType,webUrl"
    response = requests.get(endpoint, headers=headers, timeout=30)
    response.raise_for_status()
    return response.json()["value"]


if ACCESS_TOKEN:
    drives = get_site_drives(SITE_ID, HEADERS)
    for drive in drives:
        print(f"  {drive['name']:<35} id={drive['id'][:20]}...")
    DRIVE_ID = drives[0]["id"] if drives else None
else:
    print("[DEMO] Drives on site:")
    print("  Shared Documents                    id=b!abc123...")
    DRIVE_ID = "demo-drive-id"

In [ ]:
# --- Document library: download a file ---
# The file path is relative to the library root, e.g. '/Reports/Q4_Summary.xlsx'
# Power Automate's 'Get file content' action performs this exact GET request.


def download_file(site_id: str, drive_id: str, file_path: str, headers: dict) -> bytes:
    """
    Download a file from a SharePoint document library.

    Parameters
    ----------
    site_id : str
        Graph API site ID
    drive_id : str
        Drive ID for the document library
    file_path : str
        Path relative to library root, e.g. '/Reports/Q4_Summary.xlsx'
        Leading slash required.
    headers : dict
        Authorization headers

    Returns
    -------
    bytes
        Raw binary content of the file
    """
    # The /content endpoint returns a redirect to the actual download URL
    # requests follows redirects by default
    endpoint = f"{GRAPH_BASE}/sites/{site_id}/drives/{drive_id}/root:{file_path}:/content"

    # Do not send Content-Type header for file downloads
    download_headers = {"Authorization": headers["Authorization"]}

    response = requests.get(endpoint, headers=download_headers, timeout=60)
    response.raise_for_status()
    return response.content


if ACCESS_TOKEN and DRIVE_ID and DRIVE_ID != "demo-drive-id":
    file_bytes = download_file(SITE_ID, DRIVE_ID, "/Sample/test.txt", HEADERS)
    print(f"Downloaded {len(file_bytes)} bytes.")
else:
    print("[DEMO] download_file would return the binary content of the file.")
    print("Use response.content for binary files (PDF, XLSX) or response.text for text files.")

In [ ]:
# --- Document library: upload a file ---
# PUT to /root:{path}:/content for files under 4 MB.
# For larger files, use the upload session API (createUploadSession).
# Power Automate's 'Create file' action performs this PUT request.


def upload_file(
    site_id: str,
    drive_id: str,
    file_path: str,
    content: bytes,
    headers: dict,
    content_type: str = "application/octet-stream",
) -> dict:
    """
    Upload a file to a SharePoint document library (files < 4 MB).

    Parameters
    ----------
    site_id : str
        Graph API site ID
    drive_id : str
        Drive ID for the target document library
    file_path : str
        Destination path including filename, e.g. '/Reports/Q4_New.xlsx'
    content : bytes
        Binary file content to upload
    headers : dict
        Authorization headers
    content_type : str
        MIME type of the file

    Returns
    -------
    dict
        Graph API response with 'id', 'name', 'webUrl', and 'size'
    """
    endpoint = f"{GRAPH_BASE}/sites/{site_id}/drives/{drive_id}/root:{file_path}:/content"

    upload_headers = {
        "Authorization": headers["Authorization"],
        "Content-Type": content_type,
    }

    response = requests.put(endpoint, headers=upload_headers, data=content, timeout=60)
    response.raise_for_status()
    return response.json()


# Demo: upload a small text file
sample_content = b"Automation test file created via Graph API\n"

if ACCESS_TOKEN and DRIVE_ID and DRIVE_ID != "demo-drive-id":
    uploaded = upload_file(
        SITE_ID,
        DRIVE_ID,
        "/AutomationTests/graph_api_test.txt",
        sample_content,
        HEADERS,
        content_type="text/plain",
    )
    print(f"Uploaded: {uploaded['name']}")
    print(f"URL: {uploaded['webUrl']}")
    print(f"Size: {uploaded['size']} bytes")
else:
    print("[DEMO] Expected output:")
    print("Uploaded: graph_api_test.txt")
    print("URL: https://contoso.sharepoint.com/sites/Automation/...")
    print("Size: 42 bytes")

## 6. Exercises

### Exercise 6.1: Build a multi-condition filter

**Task:** Use `build_graph_filter` to construct a filter that retrieves items where:
- `Status` equals `'In Review'`, AND
- `Priority` is less than or equal to `2`

**Expected output:**
```
fields/Status eq 'In Review' and fields/Priority le 2
```

In [ ]:
# YOUR CODE HERE
# ---------------
# Use build_graph_filter() to build the two-condition filter.
# Assign the result to exercise_6_1_result.

exercise_6_1_result = None  # Replace with your answer

In [ ]:
# AUTO-GRADED TESTS — do not modify
# ----------------------------------

def test_exercise_6_1():
    assert exercise_6_1_result is not None, (
        "exercise_6_1_result is None — assign the result of build_graph_filter() to it."
    )
    assert isinstance(exercise_6_1_result, str), (
        f"Expected a string, got {type(exercise_6_1_result)}"
    )
    assert "fields/Status eq 'In Review'" in exercise_6_1_result, (
        "Filter should include: fields/Status eq 'In Review'"
    )
    assert "fields/Priority le 2" in exercise_6_1_result, (
        "Filter should include: fields/Priority le 2"
    )
    assert " and " in exercise_6_1_result, (
        "The two conditions should be joined with ' and '"
    )
    print("Exercise 6.1 passed!")
    print(f"Your filter: {exercise_6_1_result}")


test_exercise_6_1()

### Exercise 6.2: Construct a create-item request body

**Task:** Write a function `make_item_body(title, amount, status)` that returns a dict in the format required by the Graph API `create_list_item` function.

The body must be `{ "fields": { "Title": ..., "Amount": ..., "Status": ... } }` with the values from the parameters.

**Example call:**
```python
make_item_body("Q4 Budget Review", 2500.0, "Pending")
# → {"fields": {"Title": "Q4 Budget Review", "Amount": 2500.0, "Status": "Pending"}}
```

In [ ]:
# YOUR CODE HERE
# ---------------

def make_item_body(title: str, amount: float, status: str) -> dict:
    """
    Build the request body for creating a SharePoint list item via Graph API.

    Parameters
    ----------
    title : str
        Value for the Title column
    amount : float
        Value for the Amount column
    status : str
        Value for the Status column

    Returns
    -------
    dict
        Request body in Graph API format: {"fields": {...}}
    """
    # TODO: Implement this function
    pass


exercise_6_2_result = make_item_body  # Do not change this line

In [ ]:
# AUTO-GRADED TESTS — do not modify
# ----------------------------------

def test_exercise_6_2():
    assert callable(exercise_6_2_result), "make_item_body must be a function"

    result = exercise_6_2_result("Q4 Budget Review", 2500.0, "Pending")

    assert isinstance(result, dict), (
        f"Function must return a dict, got {type(result)}"
    )
    assert "fields" in result, (
        "Return value must have a top-level 'fields' key — the Graph API requires it."
    )

    fields = result["fields"]
    assert fields.get("Title") == "Q4 Budget Review", (
        f"fields['Title'] should be 'Q4 Budget Review', got {fields.get('Title')}"
    )
    assert fields.get("Amount") == 2500.0, (
        f"fields['Amount'] should be 2500.0, got {fields.get('Amount')}"
    )
    assert fields.get("Status") == "Pending", (
        f"fields['Status'] should be 'Pending', got {fields.get('Status')}"
    )

    print("Exercise 6.2 passed!")
    print(f"Your output: {result}")


test_exercise_6_2()

### Exercise 6.3: Paginate through large result sets

**Task:** The `get_list_items` function already handles pagination via `@odata.nextLink`. Write a function `count_items_matching_filter` that uses `get_list_items` to count all items matching a given filter, without loading all items into memory at once.

The function signature:
```python
def count_items_matching_filter(
    site_id: str,
    list_id: str,
    headers: dict,
    odata_filter: str,
    page_size: int = 100,
) -> int:
```

**Hint:** `get_list_items` already paginates and returns all matching items. The key insight is that you only need the **count** — so you can pass a small `top` to reduce per-page data transfer.

In [ ]:
# YOUR CODE HERE
# ---------------

def count_items_matching_filter(
    site_id: str,
    list_id: str,
    headers: dict,
    odata_filter: str,
    page_size: int = 100,
) -> int:
    """
    Count SharePoint list items matching an OData filter.

    Parameters
    ----------
    site_id : str
        Graph API site ID
    list_id : str
        GUID of the target list
    headers : dict
        Authorization headers
    odata_filter : str
        OData filter expression with fields/ prefix
    page_size : int
        Number of items to fetch per API call

    Returns
    -------
    int
        Total count of matching items across all pages
    """
    # TODO: Use get_list_items() and return len() of the result
    pass


exercise_6_3_result = count_items_matching_filter  # Do not change this line

In [ ]:
# AUTO-GRADED TESTS — do not modify
# ----------------------------------

def test_exercise_6_3():
    import inspect

    assert callable(exercise_6_3_result), "count_items_matching_filter must be a function"

    sig = inspect.signature(exercise_6_3_result)
    params = list(sig.parameters.keys())

    assert "site_id" in params, "Function must accept site_id parameter"
    assert "list_id" in params, "Function must accept list_id parameter"
    assert "odata_filter" in params, "Function must accept odata_filter parameter"

    # Verify the return type annotation or at least that the function returns an int
    # We cannot make a live call without credentials, so we verify the interface only
    return_annotation = sig.return_annotation
    assert return_annotation in (int, inspect.Parameter.empty), (
        f"Return type annotation should be int, got {return_annotation}"
    )

    print("Exercise 6.3 interface check passed!")
    print("To fully test, call count_items_matching_filter with real credentials.")


test_exercise_6_3()

In [ ]:
section_divider("Summary")

## Summary

### Key Takeaways

1. **Graph API is what Power Automate uses internally.** Every SharePoint action in Power Automate translates to a Graph API request. Understanding the API gives you insight into connector behaviour and unlocks scenarios the connector does not expose directly.

2. **Authentication uses client credentials OAuth 2.0.** Server-to-server flows authenticate as the app itself using a Client ID and Secret — no user interaction required. Tokens expire after ~1 hour and should be refreshed.

3. **OData filters in Graph API require the `fields/` prefix.** Power Automate handles this prefix automatically; when calling Graph directly you must add it manually.

4. **Document libraries are Drives.** File operations use the Drive API (`/drives/{drive-id}/root:{path}:/content`). Get a drive ID first by calling `/sites/{site-id}/drives`.

5. **Pagination is real.** Graph API returns at most 999 items per page. Always follow `@odata.nextLink` for lists that may exceed one page.

### Graph API ↔ Power Automate Connector Mapping

| Power Automate action | Graph API call |
|-----------------------|----------------|
| Get items | `GET /sites/{id}/lists/{id}/items?$expand=fields&$filter=...` |
| Get item | `GET /sites/{id}/lists/{id}/items/{item-id}?$expand=fields` |
| Create item | `POST /sites/{id}/lists/{id}/items` |
| Update item | `PATCH /sites/{id}/lists/{id}/items/{item-id}/fields` |
| Delete item | `DELETE /sites/{id}/lists/{id}/items/{item-id}` |
| Get file content | `GET /sites/{id}/drives/{id}/root:{path}:/content` |
| Create file | `PUT /sites/{id}/drives/{id}/root:{path}:/content` |

### What Is Next

The exercise file `exercises/01_sharepoint_excel_exercise.py` builds on these Graph API patterns with self-check problems covering OData query construction, column type handling, and request body formatting. Module 06 covers advanced approval flows — multi-level routing, escalation, and SLA enforcement — that build on the document library workflow from Guide 01.

In [ ]:
key_takeaways([
    "Review the key concepts covered in this notebook",
    "Practice applying these techniques to your own data",
    "Connect this material to the companion guide and slides"
])